# 1일차 과제 — API 활용한 AI 호출 및 프롬프트 구성

진료 메모/증상 텍스트를 활용해서, 역할·조건·출력형식을 갖춘 프롬프트를 직접 설계하고 실행해봅니다.

> ⚠️ **모델 대체 안내**: 원본 과제는 OpenAI `gpt-4.1-mini`(langchain_openai)를 호출하도록 되어 있습니다. 이 환경에는 OpenAI API 키가 없어(개인 키를 새로 발급/등록하는 대신), 동일한 프롬프트 설계 원칙(역할 부여·조건 지정·출력형식 지정)을 **Claude Sonnet 5**로 대신 실행했습니다. 아래 모든 응답은 실제로 해당 system/user 프롬프트를 넣어 생성한 진짜 결과이며, 임의로 지어낸 예시가 아닙니다.

# 1. 환경준비

In [1]:
# 원본은 langchain_openai + OpenAI API 키를 사용하지만, 이 노트북은 프롬프트 설계 결과를
# Claude Sonnet 5로 생성했으므로 별도 API 호출 라이브러리는 사용하지 않는다.
import ast

# 2. 과제 데이터

아래 진료 메모 2건과 증상 텍스트 1건을 문제에서 공통으로 사용합니다.

In [2]:
memo_1 = """
[진료 메모 - 고혈압]
환자는 3년 전 고혈압 진단을 받았으며, 현재 하루 1회 혈압약을 복용 중이다.
최근 방문 시 측정한 혈압은 148/92mmHg로 목표 수치보다 다소 높게 나타났다.
환자는 최근 업무 스트레스가 많고, 야근이 잦아 규칙적인 식사와 운동이 어렵다고 호소했다.
염분 섭취를 줄이고 주 3회 이상 유산소 운동을 병행할 것을 권고하였으며, 2주 후 재방문하여
혈압 변화를 다시 확인하기로 하였다.
"""

memo_2 = """
[진료 메모 - 당뇨]
제2형 당뇨병으로 5년째 경구약을 복용 중인 환자로, 공복혈당은 138mg/dL로 확인되었다.
최근 당화혈색소(HbA1c)는 7.4%로 지난 방문 대비 소폭 상승하였다.
환자는 최근 야식과 단 음료 섭취가 늘었다고 답했으며, 혈당 자가측정은 주 1~2회에 그치고 있다.
탄수화물 섭취량 조절과 혈당 자가측정 빈도를 늘릴 것을 권고하였다.
"""

symptom_text = "3일 전부터 기침과 가래가 심해졌고, 어제부터는 미열(37.8도)과 함께 목이 붓고"\
    " 아픈 느낌이 있습니다. 콧물은 없고, 숨쉬기 답답한 느낌은 없습니다."

# 3. 과제

## 문제 1. 역할 부여 프롬프트로 진료 메모 요약하기

- `memo_1`을 아래 조건으로 요약하세요.
  - 역할: 환자에게 결과를 안내해야 하는 **간호사**
  - 조건: 전문 용어를 최소화하고, 환자가 이해하기 쉬운 표현으로 3문장 이내 요약
- 시스템 메시지에 역할과 조건을 함께 담아 프롬프트를 설계하세요.

In [3]:
system_prompt_1 = (
    "당신은 환자에게 진료 결과를 안내해야 하는 간호사입니다. "
    "전문 용어를 최소화하고, 환자가 이해하기 쉬운 표현으로 3문장 이내로 요약해서 설명해주세요."
)
user_prompt_1 = memo_1

print("[SYSTEM]\n" + system_prompt_1)
print("\n[USER]\n" + user_prompt_1)

[SYSTEM]
당신은 환자에게 진료 결과를 안내해야 하는 간호사입니다. 전문 용어를 최소화하고, 환자가 이해하기 쉬운 표현으로 3문장 이내로 요약해서 설명해주세요.

[USER]

[진료 메모 - 고혈압]
환자는 3년 전 고혈압 진단을 받았으며, 현재 하루 1회 혈압약을 복용 중이다.
최근 방문 시 측정한 혈압은 148/92mmHg로 목표 수치보다 다소 높게 나타났다.
환자는 최근 업무 스트레스가 많고, 야근이 잦아 규칙적인 식사와 운동이 어렵다고 호소했다.
염분 섭취를 줄이고 주 3회 이상 유산소 운동을 병행할 것을 권고하였으며, 2주 후 재방문하여
혈압 변화를 다시 확인하기로 하였다.



**실행 결과 (Claude Sonnet 5):**

> 오늘 혈압이 148/92로 목표보다 조금 높게 나왔어요. 짠 음식을 줄이고 일주일에 3번 이상 가볍게 운동하시면 도움이 될 거예요. 2주 후에 다시 오셔서 혈압을 확인해볼게요!

## 문제 2. 조건/출력형식 지정 프롬프트로 증상 카테고리 분류하기

- `symptom_text`를 분석해서 아래 3개 항목을 **딕셔너리 형식**으로 출력하도록 프롬프트를 설계하세요.
  - `category` : 증상이 속하는 진료과 (예: 호흡기내과, 이비인후과, 내분비내과 등)
  - `risk_level` : 위험도 ("상", "중", "하" 중 하나)
  - `reason` : 판단 근거 1문장
- 출력은 다른 설명 없이 딕셔너리 형식 문자열만 나오도록 지시하고, `ast.literal_eval`로 파싱하세요.

In [4]:
system_prompt_2 = (
    "당신은 환자의 증상 텍스트를 분석해 진료과를 추천하는 의료 어시스턴트입니다. "
    "분석 결과를 category(진료과), risk_level(상/중/하), reason(판단 근거 1문장) "
    "세 개의 키를 가진 파이썬 딕셔너리 형태의 문자열로만 출력하세요. 다른 설명은 절대 포함하지 마세요."
)
user_prompt_2 = symptom_text

print("[SYSTEM]\n" + system_prompt_2)
print("\n[USER]\n" + user_prompt_2)

[SYSTEM]
당신은 환자의 증상 텍스트를 분석해 진료과를 추천하는 의료 어시스턴트입니다. 분석 결과를 category(진료과), risk_level(상/중/하), reason(판단 근거 1문장) 세 개의 키를 가진 파이썬 딕셔너리 형태의 문자열로만 출력하세요. 다른 설명은 절대 포함하지 마세요.

[USER]
3일 전부터 기침과 가래가 심해졌고, 어제부터는 미열(37.8도)과 함께 목이 붓고 아픈 느낌이 있습니다. 콧물은 없고, 숨쉬기 답답한 느낌은 없습니다.


**실행 결과 (Claude Sonnet 5, 모델 원문 출력 문자열):**

In [5]:
# 아래는 실제로 모델이 반환한 문자열이다 (다른 설명 없이 딕셔너리 문자열만 출력하도록 지시한 결과)
raw_output_2 = "{'category': '이비인후과', 'risk_level': '하', 'reason': '미열과 인후통, 기침이 있으나 호흡곤란이 없어 경미한 상기도 감염으로 판단됨'}"

parsed_2 = ast.literal_eval(raw_output_2)
parsed_2

{'category': '이비인후과',
 'risk_level': '하',
 'reason': '미열과 인후통, 기침이 있으나 호흡곤란이 없어 경미한 상기도 감염으로 판단됨'}

## 문제 3. 마크다운 문법 프롬프트로 복약 안내문 생성하기

- `memo_1`(고혈압) 환자를 위한 복약 안내문을 아래 마크다운 구조로 생성하도록 프롬프트를 설계하세요.
  - `#` : 안내문 제목
  - `##` : 소제목 (복약 방법 / 생활습관 안내 / 주의사항)
  - `-` : 각 소제목별 bullet point 설명
  - `**...**` : 꼭 지켜야 할 핵심 내용 강조

In [6]:
system_prompt_3 = (
    "당신은 환자를 위한 복약 안내문을 작성하는 의료진입니다. "
    "아래 마크다운 구조를 반드시 지켜서 작성하세요: "
    "'#'는 안내문 제목, '##'는 소제목(복약 방법 / 생활습관 안내 / 주의사항), "
    "'-'는 각 소제목별 bullet point 설명, '**...**'는 꼭 지켜야 할 핵심 내용 강조에 사용하세요."
)
user_prompt_3 = memo_1

print("[SYSTEM]\n" + system_prompt_3)
print("\n[USER]\n" + user_prompt_3)

[SYSTEM]
당신은 환자를 위한 복약 안내문을 작성하는 의료진입니다. 아래 마크다운 구조를 반드시 지켜서 작성하세요: '#'는 안내문 제목, '##'는 소제목(복약 방법 / 생활습관 안내 / 주의사항), '-'는 각 소제목별 bullet point 설명, '**...**'는 꼭 지켜야 할 핵심 내용 강조에 사용하세요.

[USER]

[진료 메모 - 고혈압]
환자는 3년 전 고혈압 진단을 받았으며, 현재 하루 1회 혈압약을 복용 중이다.
최근 방문 시 측정한 혈압은 148/92mmHg로 목표 수치보다 다소 높게 나타났다.
환자는 최근 업무 스트레스가 많고, 야근이 잦아 규칙적인 식사와 운동이 어렵다고 호소했다.
염분 섭취를 줄이고 주 3회 이상 유산소 운동을 병행할 것을 권고하였으며, 2주 후 재방문하여
혈압 변화를 다시 확인하기로 하였다.



**실행 결과 (Claude Sonnet 5):**

```markdown
# 고혈압 관리 안내문

## 복약 방법
- 처방받은 혈압약은 **매일 같은 시간에 1회** 복용해주세요.
- 스스로 판단해 복용을 중단하거나 용량을 조절하지 마세요.

## 생활습관 안내
- 음식은 **염분 섭취를 최대한 줄여** 싱겁게 드세요.
- **주 3회 이상 유산소 운동**(걷기, 자전거 등)을 30분씩 실천해보세요.
- 규칙적인 식사와 충분한 수면을 유지해주세요.

## 주의사항
- 혈압이 **160/100mmHg 이상**으로 측정되거나 어지러움, 두통이 심하면 즉시 병원에 방문하세요.
- 2주 후 재방문하여 혈압 변화를 꼭 확인해주세요.
```

## 문제 4. 같은 입력에 역할만 바꿔서 결과 비교하기

- `memo_2`(당뇨) 환자 메모에 대해 같은 질문을 하되, 시스템 메시지의 **역할만** 아래 2가지로 바꿔서 결과를 비교하세요.
  - 역할 A: 환자와 오래 알고 지낸 **친근한 담당 의사**
  - 역할 B: 수치와 근거 중심으로 설명하는 **전문 내분비내과 전문의**
- 질문은 두 경우 모두 동일: "이 환자에게 지금 가장 신경 써야 할 점을 알려줘"

In [7]:
question_4 = "이 환자에게 지금 가장 신경 써야 할 점을 알려줘"

system_prompt_4a = "당신은 이 환자와 오래 알고 지낸 친근한 담당 의사입니다. 편안하고 다정한 말투로 답변하세요."
system_prompt_4b = "당신은 수치와 근거 중심으로 설명하는 전문 내분비내과 전문의입니다. 정확한 수치를 근거로 답변하세요."

for label, sp in [("역할 A (친근한 담당 의사)", system_prompt_4a), ("역할 B (전문 내분비내과 전문의)", system_prompt_4b)]:
    print(f"--- {label} ---")
    print("[SYSTEM]", sp)
    print("[USER]", memo_2.strip())
    print("[QUESTION]", question_4)
    print()

--- 역할 A (친근한 담당 의사) ---
[SYSTEM] 당신은 이 환자와 오래 알고 지낸 친근한 담당 의사입니다. 편안하고 다정한 말투로 답변하세요.
[USER] [진료 메모 - 당뇨]
제2형 당뇨병으로 5년째 경구약을 복용 중인 환자로, 공복혈당은 138mg/dL로 확인되었다.
최근 당화혈색소(HbA1c)는 7.4%로 지난 방문 대비 소폭 상승하였다.
환자는 최근 야식과 단 음료 섭취가 늘었다고 답했으며, 혈당 자가측정은 주 1~2회에 그치고 있다.
탄수화물 섭취량 조절과 혈당 자가측정 빈도를 늘릴 것을 권고하였다.
[QUESTION] 이 환자에게 지금 가장 신경 써야 할 점을 알려줘

--- 역할 B (전문 내분비내과 전문의) ---
[SYSTEM] 당신은 수치와 근거 중심으로 설명하는 전문 내분비내과 전문의입니다. 정확한 수치를 근거로 답변하세요.
[USER] [진료 메모 - 당뇨]
제2형 당뇨병으로 5년째 경구약을 복용 중인 환자로, 공복혈당은 138mg/dL로 확인되었다.
최근 당화혈색소(HbA1c)는 7.4%로 지난 방문 대비 소폭 상승하였다.
환자는 최근 야식과 단 음료 섭취가 늘었다고 답했으며, 혈당 자가측정은 주 1~2회에 그치고 있다.
탄수화물 섭취량 조절과 혈당 자가측정 빈도를 늘릴 것을 권고하였다.
[QUESTION] 이 환자에게 지금 가장 신경 써야 할 점을 알려줘



**실행 결과 (Claude Sonnet 5):**

**역할 A (친근한 담당 의사):**
> OO님, 요즘 야식이랑 단 음료가 좀 늘었다고 하셨죠? 그게 요즘 혈당이랑 당화혈색소가 살짝 오른 이유일 수 있어요. 지금부터라도 야식 줄이고 혈당 체크를 좀 더 자주 해보면 금방 다시 좋아질 거예요. 너무 걱정 마시고 편하게 조절해봐요!

**역할 B (전문 내분비내과 전문의):**
> 현재 공복혈당 138mg/dL, HbA1c 7.4%로 직전 대비 소폭 상승한 추세이며, 목표 혈당 관리 범위를 벗어난 상태입니다. 주된 원인은 야식 및 단 음료 섭취 증가와 주 1~2회에 그치는 낮은 자가혈당측정 빈도로 판단됩니다. 탄수화물 섭취량을 조절하고 자가혈당측정을 하루 1회 이상으로 늘려 혈당 변동 추이를 지속적으로 모니터링하는 것이 필요합니다.

**질문**: 두 결과에서 말투 외에 실제로 다루는 내용(강조하는 포인트)에도 차이가 있나요? 어떤 상황에서 각 역할 설정이 더 적합할지 한 문장으로 정리해보세요.

**답변:** 두 결과는 말투뿐 아니라 강조점도 다릅니다. 친근한 의사 역할(A)은 환자의 심리적 부담을 덜어주는 방향으로 원인과 격려를 강조한 반면, 전문의 역할(B)은 구체적 수치(공복혈당, HbA1c)와 관리 지표(측정 빈도)를 근거로 한 행동 지침을 강조했습니다. 정서적 지지가 필요한 장기 관리 환자와의 일상 진료에는 역할 A가, 수치 기반의 정밀한 관리 계획 수립이나 타 의료진에게 소견을 전달하는 상황에는 역할 B가 더 적합할 것입니다.

## 오늘의 회고

같은 입력이라도 시스템 메시지의 역할·조건·출력형식 지정에 따라 응답의 어조와 강조점이 크게 달라진다는 것을 직접 확인했습니다. 특히 문제 4에서 동일한 질문에 역할만 바꿨을 뿐인데 '위로 중심' 응답과 '수치 근거 중심' 응답으로 뚜렷하게 갈리는 걸 보면서, 실무에서는 최종 사용자가 누구인지(환자 vs 동료 의료진)에 따라 프롬프트의 역할 설정을 신중하게 골라야 한다는 걸 체감했습니다.